# Nortek Aquadopp Data Processing

Processing path used: 

proc_1 IMOS NetCDF + proc_2 IMOS NetCDF -> IMOS delivery filenames

### Setup

Imports

In [1]:
import os
import sys
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

Import local tools

In [2]:
TOOLS_DIR = Path.cwd().resolve().parent
if str(TOOLS_DIR) not in sys.path:
    sys.path.insert(0, str(TOOLS_DIR))

# For IMOS NetCDF conversion
from tools.imos_nc_converter import imos_converter as imos_converter_module
importlib.reload(imos_converter_module)
IMOSNetCDFConverter_AQD = imos_converter_module.IMOSNetCDFConverter_AQD

# Read the metadata table
from tools import database_lookup as database_lookup_module
importlib.reload(database_lookup_module)
get_instrument_context = database_lookup_module.get_instrument_context
update_metadata_file_fields = database_lookup_module.update_metadata_file_fields

Definitions

In [3]:
# Working directory
os.chdir("/datasets/work/oa-srsalt/work/preqa/SWOT/cal_val/jason_calval/all_mooring_data/ash")

In [4]:
# Select instrument using ID from satellite_altimetry_moorings_metadata.csv

inst_deploy_id = 258    # rec_202502/BASS3A_PTSUV

database, _row, cfg, metadata = get_instrument_context(
    inst_deploy_id=inst_deploy_id,
    # metadata_csv="/datasets/work/oa-srsalt/work/preqa/SWOT/cal_val/jason_calval/all_mooring_data/reference_mooring_proc_info/satellite_altimetry_moorings_metadata.csv",
    print_details=True,
)

inst_deploy_ID : 258.0
site           : BASJAS | 2024f | PTSUV
position       : 145.592, -40.643
nominal_depth  : 50.0
instrument     : AQD | 5473.0 | 27.5
time_coverage  : 2024-07-31T05:25:00Z to 2025-08-23T04:15:00Z
data_in_path   : data_in/rec_202508/BASJAS_PTSUV_202508/AQD_5473
data_in_file   : PTSUV01.dat
proc_1_path    : proc_1/rec_202508/BASJAS_PTSUV_202508/AQD_5473
proc_1_file    : BASJAS_202407_AQD_5473_50m.nc
proc_2_path    : proc_2/rec_202508/BASJAS_PTSUV_202508/AQD_5473
proc_2_file    : BASJAS_202407_AQD_5473_50m.nc
imos_path      : imos_delivery/rec_202508/2024f
imos_file      : IMOS_SRSALT_PTSUV_20240715T000000Z_BASJAS_FV00_AQDd50m.nc;IMOS_SRSALT_PTSUV_20240731T052000Z_BASJAS_FV01_AQDd50m.nc


In [5]:
converter = IMOSNetCDFConverter_AQD(input_folder="", input_file="", output_dir="")

### IMOS delivery

Locate proc_1 and proc_2 files

In [6]:
def resolve_stage_dir(path_value):
    stage_dir = Path(str(path_value)).expanduser()
    if not stage_dir.is_absolute():
        stage_dir = (Path.cwd() / stage_dir).resolve()
    return stage_dir

def find_stage_file(stage_dir, stage_label, configured_name=None):
    if pd.notna(configured_name) and str(configured_name).strip():
        configured_path = stage_dir / str(configured_name).strip()
        if configured_path.exists():
            return configured_path
    pattern = f"{cfg['location']}_*_AQD_{int(cfg['inst_id'])}_*.nc"
    candidates = sorted(stage_dir.glob(pattern))
    if not candidates:
        candidates = sorted(stage_dir.glob("*.nc"))
    if not candidates:
        raise FileNotFoundError(f"No {stage_label} NetCDF files found in {stage_dir}")
    return candidates[-1]

proc_1_dir = resolve_stage_dir(cfg["proc_1_path"])
proc_2_dir = resolve_stage_dir(cfg["proc_2_path"])
proc_1_path = find_stage_file(proc_1_dir, "proc_1", configured_name=cfg.get("proc_1_file", ""))
proc_2_path = find_stage_file(proc_2_dir, "proc_2", configured_name=cfg.get("proc_2_file", ""))

In [7]:
print(f"Using proc_1 input: {proc_1_path}")
print(f"Using proc_2 input: {proc_2_path}")

Using proc_1 input: /datasets/work/oa-srsalt/work/preqa/SWOT/cal_val/jason_calval/all_mooring_data/ash/proc_1/rec_202508/BASJAS_PTSUV_202508/AQD_5473/BASJAS_202407_AQD_5473_50m.nc
Using proc_2 input: /datasets/work/oa-srsalt/work/preqa/SWOT/cal_val/jason_calval/all_mooring_data/ash/proc_2/rec_202508/BASJAS_PTSUV_202508/AQD_5473/BASJAS_202407_AQD_5473_50m.nc


In [8]:
def build_delivery_kwargs(input_path, version):
    """Build kwargs for IMOS delivery publishing using converter's time extraction."""
    with xr.open_dataset(input_path) as opened_ds:
        ds_loaded = opened_ds.load()

    attrs = dict(ds_loaded.attrs)
    if "NOMINAL_DEPTH" in ds_loaded:
        converter_depth = float(np.asarray(ds_loaded["NOMINAL_DEPTH"].values).squeeze())
    else:
        converter_depth = float(cfg.get("nominal_depth", 0.0))

    # Extract time coverage from actual data using converter method
    time_coverage_start, time_coverage_end = converter._extract_time_coverage(str(input_path))

    return {
        "input_nc_path": str(input_path),
        "longitude": float(cfg["longitude"]),
        "latitude": float(cfg["latitude"]),
        "depth": converter_depth,
        "inst_channels": attrs.get("mooring_channels", cfg.get("inst_channels", "")),
        "start_of_good_data": time_coverage_start,
        "time_deployment_start": attrs.get("time_coverage_start", cfg.get("time_coverage_start", None)),
        "time_deployment_end": attrs.get("time_coverage_end", cfg.get("time_coverage_end", None)),
        "site_code": str(cfg["location"]),
        "version": version,
        "instrument": str(cfg.get("inst_type", "")),
        "inst_id": str(int(cfg["inst_id"])),
        "location": cfg["location"],
        "output_name_mode": "imos",
        "output_stage": "imos_delivery",
        "metadata_row": _row,
        "process_mode": "publish",
        "metadata_mode": "fill_missing",
        "deployment_id": cfg.get("deployment_id", ""),
        "mooring_channels": attrs.get("mooring_channels", cfg.get("mooring_channels", "")),
        "nominal_inst_depth": cfg.get("nominal_inst_depth", ""),
        "deploy_date": cfg.get("deploy_date", ""),
        "recovery_date": cfg.get("recovery_date", ""),
        "time_coverage_start": time_coverage_start,
        "time_coverage_end": time_coverage_end,
    }

proc_1_delivery_kwargs = build_delivery_kwargs(proc_1_path, "0")
proc_2_delivery_kwargs = build_delivery_kwargs(proc_2_path, "1")

print("Ready to publish proc_1 as FV00 and proc_2 as FV01")

Ready to publish proc_1 as FV00 and proc_2 as FV01


Publish proc_1 to IMOS delivery as FV00

In [9]:
imos_fv00_out = converter.process(**proc_1_delivery_kwargs)

print(f"IMOS FV00 output: {imos_fv00_out}")

Processing BASJAS_202407_AQD_5473_50m.nc...
  Output directory: /datasets/work/oa-srsalt/work/preqa/SWOT/cal_val/jason_calval/all_mooring_data/ash/imos_delivery/rec_202508/2024f
  Publishing PTSUV channel file with delivery naming
Copied NetCDF deliverable to: IMOS_SRSALT_PTSUV_20240715T000000Z_BASJAS_FV00_AQDd50m.nc
Successfully processed BASJAS_202407_AQD_5473_50m.nc
IMOS FV00 output: /datasets/work/oa-srsalt/work/preqa/SWOT/cal_val/jason_calval/all_mooring_data/ash/imos_delivery/rec_202508/2024f/IMOS_SRSALT_PTSUV_20240715T000000Z_BASJAS_FV00_AQDd50m.nc


Publish proc_2 to IMOS delivery as FV01

In [ ]:
imos_fv01_out = converter.process(**proc_2_delivery_kwargs)

print(f"IMOS FV01 output: {imos_fv01_out}")

Write IMOS delivery filenames to metadata table

In [ ]:
imos_file_names = [Path(imos_fv00_out).name, Path(imos_fv01_out).name]
imos_file_value = ";".join(imos_file_names)
_row = update_metadata_file_fields(inst_deploy_id, {"imos_deliverables_file": imos_file_value}, output_paths={"imos_deliverables_file": imos_fv01_out}, working_dir=Path.cwd())
print(f"Updated imos_deliverables_file: {_row['imos_deliverables_file']}")

In [ ]:
# Select instrument using ID from satellite_altimetry_moorings_metadata.csv

inst_deploy_id = 258    # rec_202502/BASS3A_PTSUV

database, _row, cfg, metadata = get_instrument_context(
    inst_deploy_id=inst_deploy_id,
    # metadata_csv="/datasets/work/oa-srsalt/work/preqa/SWOT/cal_val/jason_calval/all_mooring_data/reference_mooring_proc_info/satellite_altimetry_moorings_metadata.csv",
    print_details=True,
)